**Part 1 · AI Fundamentals** · step 3 of 23  
[← 02 From LLMs to agents](02_from_llms_to_agents.ipynb) · [🏠 Roadmap](../README.md) · [04 Chat models & providers →](../02_LangChain/04_chat_models_and_providers.ipynb)

# 03 · Python Prerequisites — Pydantic & Type Hints

LangChain and LangGraph lean heavily on two Python features. Learn them once here:
- **Pydantic models** → tool arguments, structured output (notebooks 07, 12)
- **Type hints** (`TypedDict`, `Literal`, `Annotated`) → LangGraph state and routing (notebooks 16–17)

## Part A · Pydantic

**Pydantic** = Python classes with type hints that are **validated at runtime**.
LangChain uses it for tool schemas and structured output; LangGraph can use it for graph state.

| | `@dataclass` | `pydantic.BaseModel` |
|---|---|---|
| Type hints | documentation only | **enforced / converted** |
| Wrong type | silently accepted | `ValidationError` |
| Constraints (`gt`, `max_length`…) | ❌ | ✅ via `Field` |
| JSON export | manual | `.model_dump()`, `.model_dump_json()` |

*This notebook runs fully offline — no API key needed.*

## 1. Basic model

In [1]:
from pydantic import BaseModel, ValidationError


class Person(BaseModel):
    """A person with a name, age and city — all required."""

    name: str
    age: int
    city: str


person = Person(name="krish", age=35, city="blr")
person

Person(name='krish', age=35, city='blr')

In [2]:
# Wrong type -> Pydantic refuses it
try:
    Person(name="krish", age="thirty five", city="blr")
except ValidationError as e:
    print(e)

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing


## 2. Compare with `@dataclass` — no validation

In [ ]:
from dataclasses import dataclass


@dataclass
class Person1:
    """Same fields as Person, but type hints are NOT enforced."""

    name: str
    age: int
    city: str


Person1(name="krish", age=35, city=1)  # city=1 is accepted even though it should be str

## 3. Optional fields & defaults
- `Optional[type]` — the field may be `None`.
- A default value (`= None`, `= True`) makes the field optional to pass.
- Required fields must still be provided, and provided values are still validated.

In [ ]:
from typing import Optional


class Employee(BaseModel):
    """Employee record — salary and is_active have defaults."""

    id: int
    name: str
    department: str
    salary: Optional[float] = None
    is_active: Optional[bool] = True


print(Employee(id=1, name="john", department="IT"))

## 4. Lists and nested models

In [ ]:
class ClassRoom(BaseModel):
    """A list field: every item must be a str."""

    room_number: str
    students: list[str]
    capacity: int


print(ClassRoom(room_number="A101", students=["Alice", "Bob", "Marley"], capacity=30))

In [ ]:
class Address(BaseModel):
    """Nested model used inside Customer."""

    street: str
    city: str
    pincode: int


class Customer(BaseModel):
    """A model that contains another model."""

    customer_id: int
    name: str
    address: Address


customer = Customer(
    customer_id=1,
    name="Bob",
    address={"street": "Motihari", "city": "Patna", "pincode": 12345},  # dicts are converted automatically
)
customer.model_dump()

## 5. Field validation with `Field`
`gt`/`ge`/`lt`/`le` for numbers, `min_length`/`max_length` for strings and lists,
`description` for docs (and for the LLM when used in structured output).

In [ ]:
from pydantic import Field


class Item(BaseModel):
    """An item with validated name length, price range and quantity."""

    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0, le=1000)
    quantity: int = Field(ge=0)


print(Item(name="Ram", price=30, quantity=5))

try:
    Item(name="R", price=-1, quantity=5)
except ValidationError as e:
    print(e)

## Part B · Type hints you'll meet in LangGraph

| Hint | Meaning | Where it's used |
|---|---|---|
| `TypedDict` | a dict with fixed, typed keys (**no** runtime validation) | LangGraph **state** |
| `Literal["a", "b"]` | the value must be one of these exact strings | router return types |
| `Annotated[T, extra]` | type `T` plus extra metadata for a library to read | **reducers** (`add_messages`) |

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict


class ChatState(TypedDict):
    """A TypedDict is just a normal dict at runtime — the hints are for you and your tools."""

    user: str
    turns: Annotated[int, "between 0 and 10"]


state: ChatState = {"user": "Ujjawal", "turns": 0}
print(type(state), state)


def route(state: ChatState) -> Literal["greet", "help"]:
    """A router returns one of a fixed set of names (this is how LangGraph picks the next node)."""
    return "greet" if state["turns"] == 0 else "help"


print(route(state))


# Annotated attaches metadata; Python ignores it, libraries like LangGraph read it
Score = Annotated[int, "between 0 and 10"]
print(Score.__metadata__)

<class 'dict'> {'user': 'Ujjawal', 'turns': 0}
greet
('between 0 and 10',)


## Revision notes
- Pydantic `BaseModel` → **validated** data (tool args, structured output).
- `TypedDict` → typed dict, **not** validated (graph state).
- `Literal` → fixed choices; `Annotated` → type + metadata (reducers, notebook 17).

---
✅ **Done with 03 Python prerequisites.** Next: [04 Chat models & providers →](../02_LangChain/04_chat_models_and_providers.ipynb)  
[🏠 Roadmap](../README.md)